# DPN Classification - YOLOv11 Training (Google Colab)

Trains YOLOv11 on a free T4 GPU - ~15 min vs 3+ hrs on CPU.

## Before running:
1. **Enable GPU**: Runtime -> Change runtime type -> T4 GPU -> Save
2. **Run all**: Runtime -> Run all

## After training:
- best_yolo_model.pt is saved to Google Drive under DPN_Checkpoints/
- Download it, place in checkpoints/ on your PC, restart the API.

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU - go to Runtime -> Change runtime type -> T4 GPU -> Save')


In [ ]:
import sys, subprocess, shutil
from pathlib import Path

PROJECT_DIR = '/content/dpn_project'
GITHUB_REPO = 'https://github.com/catnipp9/transistors-thermal-ai-testing.git'

if Path(PROJECT_DIR).exists():
    shutil.rmtree(PROJECT_DIR)

result = subprocess.run(['git', 'clone', GITHUB_REPO, PROJECT_DIR], capture_output=True, text=True)
if result.returncode != 0:
    print('Clone FAILED:', result.stderr)
    raise RuntimeError('Could not clone repo')

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from models.data_loader import prepare_yolo_dataset
from models.model import YOLOv11DPNClassifier
from models.trainer import YOLOTrainer
print('Repo cloned and modules loaded.')

log = subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True, cwd=PROJECT_DIR)
print('Latest commits:')
print(log.stdout)


In [ ]:
!pip install ultralytics>=8.3.0 scikit-learn>=1.4.0 scipy>=1.12.0 pyyaml>=6.0.0 gdown --quiet
print('Packages installed.')


In [ ]:
# Download dataset directly from shared Google Drive folder using folder ID
# This works for shared folders - no MyDrive access needed
import gdown
from pathlib import Path

FOLDER_ID     = '1zI5gmI57iBcVgbMxWzrwOKAz5k3jrKgh'
DRIVE_DATASET = '/content/ThermoDataBase'

if not Path(DRIVE_DATASET).exists():
    print('Downloading dataset from Google Drive...')
    gdown.download_folder(id=FOLDER_ID, output=DRIVE_DATASET, quiet=False, use_cookies=False)
else:
    print('Dataset already downloaded, skipping.')

control_dir = Path(DRIVE_DATASET) / 'Control Group'
dm_dir      = Path(DRIVE_DATASET) / 'DM Group'
control_count = len([d for d in control_dir.iterdir() if d.is_dir()]) if control_dir.exists() else 0
dm_count      = len([d for d in dm_dir.iterdir() if d.is_dir()]) if dm_dir.exists() else 0

print(f'Control Group : {control_count} subjects')
print(f'DM Group      : {dm_count} subjects')

if control_count == 0 and dm_count == 0:
    print('\nContents of downloaded folder:')
    for item in sorted(Path(DRIVE_DATASET).iterdir()):
        print(f'  {item.name}')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from google.colab import drive

# Mount Drive just for saving the trained model
drive.mount('/content/drive', force_remount=False)

CHECKPOINT_DIR = '/content/drive/MyDrive/DPN_Checkpoints'
YOLO_DATASET   = '/content/yolo_dataset'

Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

print(f'Dataset       : {DRIVE_DATASET}')
print(f'Checkpoints   : {CHECKPOINT_DIR}')
print(f'PyTorch       : {torch.__version__}')
print(f'CUDA          : {torch.cuda.is_available()}')


In [ ]:
# Build YOLO dataset with oversampling to fix class imbalance
yaml_path = prepare_yolo_dataset(
    data_dir=DRIVE_DATASET,
    output_dir=YOLO_DATASET,
)
print(f'Dataset ready: {yaml_path}')


## Training Settings

| Setting | Value | Notes |
|---------|-------|-------|
| `YOLO_VARIANT` | `yolo11l-cls` | Large model - best for GPU |
| `EPOCHS` | `100` | More = better accuracy |
| `PATIENCE` | `20` | Early stopping patience |
| `BATCH` | `32` | Larger batch works well on GPU |

In [ ]:
# Edit these settings to improve accuracy
YOLO_VARIANT = 'yolo11l-cls'
EPOCHS       = 100
PATIENCE     = 20
IMGSZ        = 224
BATCH        = 32
LR           = 0.001

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Variant : {YOLO_VARIANT}')
print(f'Epochs  : {EPOCHS} (patience={PATIENCE})')
print(f'Device  : {device_name}')

yolo_model   = YOLOv11DPNClassifier(variant=YOLO_VARIANT)
yolo_trainer = YOLOTrainer(yolo_model, save_dir=CHECKPOINT_DIR)

yolo_history = yolo_trainer.train(
    data_yaml=yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    lr0=LR,
    patience=PATIENCE,
)
print('Training complete!')
print(f'Results: {yolo_history}')


In [ ]:
# Save best weights to Google Drive
yolo_trainer.save_best_checkpoint()

saved_path = Path(CHECKPOINT_DIR) / 'best_yolo_model.pt'
if saved_path.exists():
    size_mb = saved_path.stat().st_size / 1e6
    print(f'Model saved: {saved_path} ({size_mb:.1f} MB)')
    print('Find it in MyDrive/DPN_Checkpoints/ on your Google Drive')
else:
    print('WARNING: model file not found - check training completed successfully')


In [ ]:
# Download model directly to your PC
from google.colab import files
model_path = Path(CHECKPOINT_DIR) / 'best_yolo_model.pt'
if model_path.exists():
    files.download(str(model_path))
    print('Download started. Place the file in checkpoints/ on your PC.')
    print('Then restart the API: uvicorn api.main:app --reload')
else:
    print('Model not found - make sure training completed.')
